In [ ]:
import pandas as pd

# Load CSV files
users = pd.read_csv("Users.csv")
products = pd.read_csv("Products.csv")
orders = pd.read_csv("Orders.csv")
order_items = pd.read_csv("Order_Items.csv")
payments = pd.read_csv("Payments.csv")
reviews = pd.read_csv("Reviews.csv")
offers = pd.read_csv("Offers.csv")

# Revenue
order_items["revenue"] = order_items["quantity"] * order_items["unit_price"]

# Merge 1
sales = pd.merge(
    order_items,
    products,
    left_on="product_id",
    right_on="id",
    how="left",
    suffixes=("", "_product")
)

# Merge 2
sales = pd.merge(
    sales,
    orders,
    on="invoice_no",
    how="left"
)

# Merge 3
sales = pd.merge(
    sales,
    users,
    left_on="customer_id",
    right_on="id",
    how="left",
    suffixes=("", "_user")
)

# Merge 4
sales = pd.merge(
    sales,
    payments,
    on="invoice_no",
    how="left",
    suffixes=("", "_payment")
)

print(sales.columns)
print(sales.head())

Index(['id', 'invoice_no', 'product_id', 'quantity', 'unit_price', 'revenue',
       'id_product', 'name', 'price', 'sales_count', 'view_count', 'rating',
       'category', 'customer_id', 'order_date', 'id_user', 'country',
       'id_payment', 'payment_method', 'payment_status', 'payment_date'],
      dtype='object')
   id invoice_no product_id  quantity  unit_price  revenue id_product  \
0   1     ORD001       P001         2       250.0    500.0       P001   
1   2     ORD001       P003         5        15.0     75.0       P003   
2   3     ORD002       P002         6        25.0    150.0       P002   
3   4     ORD002       P004         2        40.0     80.0       P004   
4   5     ORD003       P005         4        35.0    140.0       P005   

                  name  price  sales_count  ...  rating   category  \
0  Nescafe Gold Coffee  250.0          150  ...     4.9  Beverages   
1      Lays Salt Chips   15.0          450  ...     4.6     Snacks   
2             Pepsi 1L   25.0 

In [ ]:
# ==============================
# KPI ENGINE
# ==============================

# Revenue
total_revenue = sales["revenue"].sum()

completed_revenue = sales.loc[
    sales["payment_status"] == "Completed",
    "revenue"
].sum()

# Orders
total_orders = orders["invoice_no"].nunique()

avg_order_value = total_revenue / total_orders

completion_rate = (
    payments["payment_status"].eq("Completed").mean() * 100
)

# Products
top_products = (
    sales.groupby("name")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

weak_products = (
    sales.groupby("name")["quantity"]
    .sum()
    .sort_values()
    .head(5)
)

# Categories
category_revenue = (
    sales.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Countries
country_revenue = (
    sales.groupby("country")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Daily Revenue
daily_revenue = (
    sales.groupby(
        pd.to_datetime(sales["order_date"]).dt.date
    )["revenue"]
    .sum()
)

# Payment
payment_methods = payments["payment_method"].value_counts()

payment_status = payments["payment_status"].value_counts()

# Reviews
average_rating = reviews["rating"].mean()

top_rated = (
    reviews.groupby("product_id")["rating"]
    .mean()
    .sort_values(ascending=False)
)

# Offers
active_offers = offers["is_active"].sum()

# ==============================
# PRINT RESULTS
# ==============================

print("="*60)
print("HYPERMARKET KPI REPORT")
print("="*60)

print(f"Total Revenue       : ${total_revenue:.2f}")
print(f"Completed Revenue   : ${completed_revenue:.2f}")
print(f"Total Orders        : {total_orders}")
print(f"Average Order Value : ${avg_order_value:.2f}")
print(f"Completion Rate     : {completion_rate:.2f}%")
print(f"Average Rating      : {average_rating:.2f}")
print(f"Active Offers       : {active_offers}")

print("\n========== Top Products ==========")
print(top_products)

print("\n========== Weak Products ==========")
print(weak_products)

print("\n========== Category Revenue ==========")
print(category_revenue)

print("\n========== Country Revenue ==========")
print(country_revenue)

print("\n========== Payment Methods ==========")
print(payment_methods)

print("\n========== Payment Status ==========")
print(payment_status)

print("\n========== Top Rated Products ==========")
print(top_rated)

print("\n========== Daily Revenue ==========")
print(daily_revenue)

HYPERMARKET KPI REPORT
Total Revenue       : $5314.00
Completed Revenue   : $4784.00
Total Orders        : 20
Average Order Value : $265.70
Completion Rate     : 90.00%
Average Rating      : 4.65
Active Offers       : 10

========== Top Products ==========
name
Nescafe Gold Coffee    750.0
Nutella 350G           360.0
Olive Oil 500ML        360.0
Rice 1KG               350.0
Cheese Spread          300.0
Name: revenue, dtype: float64

========== Weak Products ==========
name
Butter 250G       1
Chocolate Cake    1
Corn Flakes       1
Frozen Burger     1
Frozen Pizza      1
Name: quantity, dtype: int64

========== Category Revenue ==========
category
Beverages      1652.0
Groceries      1216.0
Chocolate       705.0
Snacks          674.0
Dairy           507.0
Bakery          290.0
Frozen Food     270.0
Name: revenue, dtype: float64

========== Country Revenue ==========
country
Egypt           860.0
Jordan          800.0
France          580.0
Kuwait          536.0
Oman            534.0
Qa

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def generate(prompt):

    messages = [
        {
            "role": "system",
            "content": """
You are a Senior Business Intelligence Consultant.

Write executive reports in McKinsey/BCG consulting style.

Rules:
- No repetition of KPI tables
- No generic filler sentences
- Deep interpretation only
- Structured sections with clear headings
- Always complete the report fully
- Be concise but insightful
"""
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = pipe(
        text,
        max_new_tokens=1200,   # مهم
        do_sample=False,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )

    return output[0]["generated_text"]

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.7 MB/s eta 0:00:00


In [ ]:
prompt = """
You are a Senior Business Intelligence Analyst.

You MUST follow these rules strictly:
- Use ONLY the provided numbers.
- Do NOT generate trends (YoY, growth, increase) unless explicitly given.
- Every insight MUST include a number or computed comparison from the data.
- No vague business statements.
- No assumptions.


BUSINESS DATA


Total Revenue: 5314
Completed Revenue: 4784
Completion Rate: 90%
Average Order Value: 265.70
Average Rating: 4.65

CATEGORY REVENUE:
Beverages: 1652
Groceries: 1216
Chocolate: 705
Snacks: 674
Dairy: 507
Bakery: 290
Frozen Food: 270

TOP PRODUCTS (Revenue):
Nescafe Gold Coffee: 750
Nutella 350G: 360
Olive Oil 500ML: 360
Rice 1KG: 350
Cheese Spread: 300

WEAK PRODUCTS (Quantity):
Butter 250G: 1
Chocolate Cake: 1
Corn Flakes: 1
Frozen Burger: 1
Frozen Pizza: 1

COUNTRY REVENUE:
Egypt: 860
Jordan: 800
France: 580
Kuwait: 536
Oman: 534
Qatar: 480
UAE: 444
Bahrain: 420
Saudi Arabia: 414
Germany: 246

PAYMENTS:
Completed: 18
Pending: 2

ANALYSIS RULES


1. Always compare values (e.g., Egypt vs Germany difference = X).
2. Always rank (Top 1, Top 2... clearly).
3. Always quantify gaps (difference or percentage if possible).
4. Never use generic phrases like "strong performance".
5. Every insight must reference a number explicitly.


OUTPUT STRUCTURE


Executive Summary
- ONLY 2 paragraphs
- Must include 2–3 numeric insights

Financial Performance
- Break down revenue vs completed revenue (difference must be calculated)
- Explain completion rate impact using numbers
- AOV interpretation with reasoning

Product Performance
- Rank top products with gaps between them
- Identify weakest products and quantify issue (e.g., all = 1 unit)
- Compare category dominance (difference between top and second)

Regional Performance
- Rank countries
- Show numeric gaps (e.g., Egypt - Germany = X)
- Identify concentration risk if top 2 > X%

Risks
- Must be based on numeric thresholds only

Recommendations
- 5–7 actionable ONLY
- Each recommendation must reference a KPI or number

Conclusion
- Data-driven final assessment
"""

In [ ]:
report = generate(prompt)

print("="*80)
print("HYPERMARKET EXECUTIVE REPORT")
print("="*80)
print(report)
print("="*80)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up

HYPERMARKET EXECUTIVE REPORT
### Executive Summary

The company's financial performance indicates a robust overall revenue of $5,314, with $4,784 in completed sales, representing a healthy completion rate of 90%. The average order value stands at $265.70, suggesting that while customer satisfaction is high (average rating of 4.65), there is potential for increasing the average transaction size to enhance profitability further. 

### Financial Performance

**Revenue vs Completed Revenue:**  
The total revenue of $5,314 contrasts sharply with the completed revenue of $4,784, indicating an uncompleted revenue of $530 ($5,314 - $4,784). This suggests that nearly 10% of orders placed did not result in payment, which could be due to various reasons such as abandoned carts or disputes over product quality.

**Completion Rate Impact:**  
A 90% completion rate implies that out of every 100 orders, 90 are successfully converted into payments. If this rate were to improve by just 5%, it would tra

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
with open("/content/drive/MyDrive/report.txt", "w", encoding="utf-8") as f:
    f.write(report)

In [ ]:

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import inch
from reportlab.lib.pagesizes import A4

def add_footer(canvas, doc):
    canvas.saveState()

    canvas.setFont("Helvetica", 9)
    canvas.drawString(40, 20, "Business Intelligence Department")
    canvas.drawRightString(550, 20, f"Page {doc.page}")

    canvas.restoreState()
def save_pdf(report_text,
             total_revenue,
             completed_revenue,
             completion_rate,
             avg_order_value,
             average_rating):

    doc = SimpleDocTemplate(
        "Executive_Report.pdf",
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=60,
        bottomMargin=40,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "TitleStyle",
        parent=styles["Title"],
        fontName="Helvetica-Bold",
        fontSize=22,
        alignment=TA_CENTER,
        textColor=colors.black,
        spaceAfter=20,
    )

    heading_style = ParagraphStyle(
        "HeadingStyle",
        parent=styles["Heading1"],
        fontName="Helvetica-Bold",
        fontSize=15,
        textColor=colors.black,
        spaceBefore=18,
        spaceAfter=10,
    )

    sub_heading_style = ParagraphStyle(
        "SubHeadingStyle",
        parent=styles["BodyText"],
        fontName="Helvetica-Bold",
        fontSize=12,
        textColor=colors.HexColor("#555555"),
        spaceBefore=10,
        spaceAfter=6,
    )

    body_style = ParagraphStyle(
        "BodyStyle",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=11,
        leading=20,
        textColor=colors.black,
        spaceAfter=10,
        leftIndent=15,
    )


    bullet_style = ParagraphStyle(
        "BulletStyle",
        parent=body_style,
        leftIndent=18,
        bulletIndent=8,
    )

    story = []

    story.append(Spacer(1, 0.3 * inch))
    story.append(Paragraph("HYPERMARKET", title_style))
    story.append(Paragraph("Executive Business Report", heading_style))
    story.append(Paragraph("Prepared for: Chief Executive Officer (CEO)", body_style))
    story.append(Paragraph("Prepared by: Business Intelligence Department", body_style))
    story.append(Spacer(1, 0.35 * inch))

    kpi_data = [
        ["KPI", "Value"],
        ["Total Revenue", f"${total_revenue:.2f}"],
        ["Completed Revenue", f"${completed_revenue:.2f}"],
        ["Completion Rate\n", f"{completion_rate:.2f}%"],
        ["Average Order Value", f"${avg_order_value:.2f}"],
        ["Average Rating", f"{average_rating:.2f}"],
    ]

    table = Table(kpi_data, colWidths=[250, 150])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9D9D9")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BACKGROUND", (0, 1), (-1, -1), colors.HexColor("#F5F5F5")),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#BFBFBF")),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
        ("ALIGN", (1, 1), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    story.append(table)
    story.append(Spacer(1, 0.4 * inch))

    for line in report_text.split("\n"):
        line = line.strip().replace("**", "")

        if not line:
            story.append(Spacer(1, 8))
            continue

        if line.startswith("#"):
            story.append(Paragraph(line.replace("#", "").strip(), heading_style))
            continue

        if line.endswith(":"):
            from reportlab.platypus import KeepTogether

            story.append(
                KeepTogether([
                      Paragraph(line, body_style)
    ])
)
            story.append(Spacer(1, 8))   # مسافة بعد العنوان الفرعي
            continue

        if line.startswith("-"):
            story.append(
                Paragraph(
                    line[1:].strip(),
                    bullet_style,
                    bulletText="•"
                )
            )
            continue

        story.append(Paragraph(line, body_style))

    doc.build(
        story,
        onFirstPage=add_footer,
        onLaterPages=add_footer
    )

    print("Executive_Report.pdf created successfully")
# Example:
# save_pdf(report, total_revenue, completed_revenue, completion_rate,
#          avg_order_value, average_rating)


In [ ]:
save_pdf(report, total_revenue, completed_revenue, completion_rate, avg_order_value, average_rating)

Executive_Report.pdf created successfully


In [ ]:
from google.colab import files

files.download("Executive_Report.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>